In [1]:
import sys
import os
import numpy as np

sys.path.append(os.path.abspath('..'))

from src.config import SolverConfig
from src.models.spacecraft import Spacecraft
from src.models.body import Body
from src.models.orbit import Orbit_Eq
from src.optimizer import Optimizer
from src.utils import cart2eq
from src.planet_propagator import earth_state, mars_state

In [ ]:
t0 = 0.0

def run_trajectory(opt, t0, tf, y0):
    sol = opt.propagator.forward(
        y0=y0,
        tspan=(t0, tf),
        control=opt.qlaw_control()
    )

    return sol

# outer loooooop

tf_values = np.linspace(120, 300, 10) * 24 * 3600  # 120–300 days
best_tf = None
best_cost = 1e99
best_sol = None

for tf in tf_values:

    print("testing tf ", tf)

    Earth = Body(name="earth",mu=3.986e14)
    Mars = Body(name="mars",mu=4.283e13)
    Sun = Body(name="sun",mu=1.327e20)

    Psyche = Spacecraft()

    r_E, v_E = earth_state(t0)

    # Earth parking orbit (simple circular approx)
    r_parking_E = 6678e3
    v_parking_E = np.sqrt(Earth.mu / r_parking_E)

    # Heliocentric escape from Earth parking orbit
    v_escape_earth = np.sqrt(2 * Earth.mu / r_parking_E)  # ~10.85 km/s
    v_earth_orbit = np.linalg.norm(v_E)                   # ~29.78 km/s

    # Hyperbolic escape tangent to Earth velocity
    delta_v_escape = v_escape_earth - v_parking_E         # ~3.13 km/s
    v0 = v_E + np.array([0, delta_v_escape, 0])           # Heliocentric departure!
    r0 = r_E + np.array([r_parking_E * 0.9, 0, 0])        # Slightly outside Earth radius

    r0 = r_E + np.array([1e8, 0, 0])  # 100,000 km from Earth center
    v0 = v_E + np.array([0, 3.13e3, 0])  # +3.13 km/s escape delta-v
    
    m0 = Psyche.m0

    y0 = np.hstack((r0, v0, m0))

    r_M, v_M = mars_state(tf)

    mee_target = cart2eq(np.hstack((r_M, v_M)))[0:5]
    cfg = SolverConfig()
    opt = Optimizer(cfg, Psyche, target_orbit=mee_target)

    sol = run_trajectory(opt, t0, tf, y0)

    yf = sol.y[:, -1]

    rf = yf[0:3]
    vf = yf[3:6]
    rrel = rf - r_M
    vrel = vf - v_M

    # terminal error
    pos_err = np.linalg.norm(rrel)
    vel_err = np.linalg.norm(vrel)

    mass = yf[6]

    cost = pos_err + 1e3 * vel_err +10*abs(rrel[2]) + 10*abs(vrel[2]) - 1e-6 * mass + 1e-6 * tf
    print(cost)

    if cost < best_cost:
        best_cost = cost
        best_tf = tf
        best_sol = sol

print("Best TOF [days]:", best_tf / 86400)
print("Best cost:", best_cost)

sol = best_sol
print("Final state:", sol.y[:, -1])

testing tf  10368000.0
140565053425.22232
testing tf  12096000.0
158918306148.8237
testing tf  13824000.0
175067118563.79758
testing tf  15552000.0
187741559626.05225
testing tf  17280000.0
196331833960.31488
testing tf  19008000.0
200518247962.13126
testing tf  20736000.0
200191141520.99356
testing tf  22464000.0
195432084244.15762
testing tf  24192000.0
186411052390.07346
testing tf  25920000.0
173492445367.5431
Best TOF [days]: 120.0
Best cost: 140565053425.22232
Final state: [-6.08805878e+10  1.82174390e+11 -2.30574177e+09 -2.56970589e+04
 -3.15629696e+03 -2.76774964e+02  2.59203443e+03]


In [3]:
mars_r, mars_v = mars_state(best_tf)
print("Final spacecraft z:", best_sol.y[2, -1])
print("Mars z:", mars_r[2])
print("Final spacecraft vz:", best_sol.y[5, -1])
print("Mars vz:", mars_v[2])

Final spacecraft z: -2305741773.9249096
Mars z: 5773944906.012167
Final spacecraft vz: -276.7749636435865
Mars vz: 536.3725622677802


In [4]:
# Jupyter Notebook Cell - Plot Best Trajectory
import plotly.graph_objects as go
import numpy as np
import json

# Get planet states
earth_initial, earth_initial_v = earth_state(t0)
mars_initial, _ = mars_state(t0)
mars_final, mars_final_v = mars_state(best_tf)

# Extract trajectory data
t_days = best_sol.t / 86400  # Convert to days
r_traj = best_sol.y[0:3, :].T / 1e9  # Position in Gm (Gigameters)
Sun_pos = np.zeros((len(t_days), 3))

# Create 3D trajectory plot
fig = go.Figure()

# 1. Spacecraft trajectory
fig.add_trace(go.Scatter3d(
    x=r_traj[:, 0], y=r_traj[:, 1], z=r_traj[:, 2],
    mode='lines',
    name='Spacecraft Trajectory',
    line=dict(width=5, color='cyan'),
    hovertemplate='<b>Time: %{text:.1f} days</b><br>X: %{x:.2f} Gm<br>Y: %{y:.2f} Gm<br>Z: %{z:.2f} Gm<extra></extra>',
    text=t_days
))

# 2. Earth initial position
fig.add_trace(go.Scatter3d(
    x=[earth_initial[0]/1e9], y=[earth_initial[1]/1e9], z=[earth_initial[2]/1e9],
    mode='markers+text',
    name='Earth (Start)',
    marker=dict(size=12, color='blue'),
    text=['Earth Start'],
    textposition="middle center",
    hovertemplate='<b>Earth Initial</b><br>Position: (%{x:.2f}, %{y:.2f}, %{z:.2f}) Gm<extra></extra>'
))

# 2. Mars initial position
fig.add_trace(go.Scatter3d(
    x=[mars_initial[0]/1e9], y=[mars_initial[1]/1e9], z=[mars_initial[2]/1e9],
    mode='markers+text',
    name='Mars (Start)',
    marker=dict(size=12, color='pink'),
    text=['Mars Start'],
    textposition="middle center",
    hovertemplate='<b>Mars Initial</b><br>Position: (%{x:.2f}, %{y:.2f}, %{z:.2f}) Gm<extra></extra>'
))

# 3. Mars final position  
fig.add_trace(go.Scatter3d(
    x=[mars_final[0]/1e9], y=[mars_final[1]/1e9], z=[mars_final[2]/1e9],
    mode='markers+text',
    name='Mars (Target)',
    marker=dict(size=12, color='red'),
    text=['Mars Target'],
    textposition="middle center",
    hovertemplate='<b>Mars Target</b><br>Position: (%{x:.2f}, %{y:.2f}, %{z:.2f}) Gm<extra></extra>'
))

# 4. Sun at origin
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers',
    name='Sun',
    marker=dict(size=15, color='yellow', symbol='circle', line=dict(color='orange', width=2))
))

# Layout
fig.update_layout(
    title=dict(
        text="Earth-Mars Low-Thrust Transfer Trajectory<br><sup>TOF: {:.1f} days | Final Distance Error: {:.1f} Mm</sup>".format(
            best_tf/86400, np.linalg.norm(best_sol.y[0:3,-1] - mars_final)
        ),
        x=0.05,
        xanchor='left'
    ),
    scene=dict(
        xaxis_title="X (Gm)",
        yaxis_title="Y (Gm)", 
        zaxis_title="Z (Gm)",
        aspectmode='cube',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
    ),
    showlegend=True,
    height=700,
    hovermode='closest'
)

# Show interactive plot in Jupyter
fig.show()

# Print key stats
print(f"🚀 Mission Summary:")
print(f"   Duration: {best_tf/86400:.1f} days")
print(f"   Final pos error: {np.linalg.norm(best_sol.y[0:3,-1] - mars_final)/1e6:.1f} Mm")
print(f"   Final vel error: {np.linalg.norm(best_sol.y[3:6,-1] - mars_final_v)/1e3:.1f} km/s")
print(f"   Final mass: {best_sol.y[6,-1]:.0f} kg")
print(f"   Delta-V used: {(m0 - best_sol.y[6,-1])/m0*100:.1f}% of initial mass")

🚀 Mission Summary:
   Duration: 120.0 days
   Final pos error: 59764.9 Mm
   Final vel error: 3.2 km/s
   Final mass: 2592 kg
   Delta-V used: 5.2% of initial mass


In [5]:
print(best_sol.y[0:3,-1])

[-6.08805878e+10  1.82174390e+11 -2.30574177e+09]
